# 实验 4：中文论文补充实验

本 Notebook 直接沿用实验 3 的 AutoDL 环境和项目调用方式，不引入新的第三方依赖。

目录结构按当前服务器设置：

```text
/root/
├── run_experiment.py
├── src/
├── dataset/
├── notebooks/
├── results/
└── autodl-tmp/
```

统一实验配置：

- 最大输入长度：512
- 掩码率：0.15
- AHP 候选数：50
- AHP 温度：1.0
- AHP 剪枝：none
- 聚合方法：majority_vote
- SelfDenoise 集成数：50，去噪器：roberta
- Top-K 集成数：50
- 随机种子：42、123、666
- SST-2：validation，完整规模 872 条
- AG News：test，完整规模 1000 条
- 查询预算：SST-2 为 100，AG News 为 200

## 运行原则

1. 先运行环境单元和检查单元。
2. 所有实验开关默认均为 `False`，不会自动运行。
3. 先保持 `EXPERIMENT_SCALE = "smoke"` 做少量测试。
4. 测试通过后再改为 `EXPERIMENT_SCALE = "full"`。
5. 不建议直接使用 `Run All`。

In [1]:
# 与实验 3 保持一致的环境
import os
import textattack

os.environ["HF_HOME"] = "/root/autodl-tmp/cache/huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/root/autodl-tmp/cache/huggingface"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE_IPYWIDGETS"] = "1"

print(f"--- 强制设置 HF 缓存路径为: {os.environ['HF_HOME']} ---")

import sys
import pandas as pd
import subprocess
import logging
from IPython.display import display
import gc
import random

# 仅使用 Python 标准库和当前环境已有的 numpy
import time
import json
import csv
import re
import difflib
import shutil
import numpy as np

# Notebook 放在 /root/notebooks 时，父目录为 /root
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    logging.info(f"已将 '{module_path}' 添加到 sys.path")

try:
    import torch
    import transformers
    import datasets
    from src.args_config import AHPSettings
    from src.experiment_runner import ExperimentRunner
    from src.utils.data_loader import load_dataset
    from src.models.model_loader import DATASET_INSTRUCTIONS
    logging.info("项目模块导入成功。")
except ImportError as e:
    logging.error(f"导入项目模块失败: {e}", exc_info=True)
    raise

2026-07-23 08:34:42.283766: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-23 08:34:42.337683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-23 08:34:43.288208: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/root/ahp_env/lib/py

--- 强制设置 HF 缓存路径为: /root/autodl-tmp/cache/huggingface ---


## 1. 全局配置与必要检查

In [2]:
PROJECT_ROOT = "/root"
MODEL_PATH = "/root/autodl-tmp/alpaca-native"
DATASET_PATH = "/root/dataset"
CACHE_DIR = "/root/autodl-tmp/cache"

RESULT_ROOT = "/root/results/joca_04"
os.makedirs(RESULT_ROOT, exist_ok=True)

# smoke：只做少量流程测试；full：论文正式实验
EXPERIMENT_SCALE = "full"

SEEDS = [123]
MAX_SEQ_LENGTH = 512
MODEL_BATCH_SIZE = 4
MASK_RATE = 0.15
ENSEMBLE_SIZE = 50

FULL_NUM_EXAMPLES = {
    "sst2": 872,
    "agnews": 1000,
}
SMOKE_NUM_EXAMPLES = {
    "sst2": 20,
    "agnews": 20,
}
FULL_ATTACK_NUM_EXAMPLES = {
    "sst2": 100,
    "agnews": 100,
}
FULL_ABLATION_NUM_EXAMPLES = 50

def get_num_examples(dataset_name):
    if EXPERIMENT_SCALE == "smoke":
        return SMOKE_NUM_EXAMPLES[dataset_name]
    if EXPERIMENT_SCALE == "full":
        return FULL_NUM_EXAMPLES[dataset_name]
    raise ValueError("EXPERIMENT_SCALE 只能是 smoke 或 full")

required_paths = [
    MODEL_PATH,
    os.path.join(DATASET_PATH, "sst2", "validation.txt"),
    os.path.join(DATASET_PATH, "agnews", "test.tsv"),
    os.path.join(PROJECT_ROOT, "src", "experiment_runner.py"),
]

missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    raise FileNotFoundError("以下路径不存在：\n" + "\n".join(missing_paths))

expected_label_tokens = {
    "sst2": [8178, 6374],
    "agnews": [2787, 12453, 15197, 17968],
}

for dataset_name, expected in expected_label_tokens.items():
    actual = DATASET_INSTRUCTIONS[dataset_name]["label_tokens"]
    print(dataset_name, "label_tokens =", actual)
    if actual != expected:
        raise ValueError(
            f"{dataset_name} 标签 Token 仍不正确：当前 {actual}，应为 {expected}"
        )

if "Science/Technology" in DATASET_INSTRUCTIONS["agnews"]["classification"]:
    raise ValueError(
        "AG News 分类提示词仍包含 Science/Technology，请改成 Technology。"
    )

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TextAttack:", getattr(textattack, "__version__", "unknown"))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("实验规模:", EXPERIMENT_SCALE)
print("结果目录:", RESULT_ROOT)

sst2 label_tokens = [8178, 6374]
agnews label_tokens = [2787, 12453, 15197, 17968]
Python: /root/ahp_env/bin/python
PyTorch: 2.8.0+cu128
Transformers: 4.57.1
TextAttack: unknown
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
实验规模: full
结果目录: /root/results/joca_04


## 2. 沿用实验 3 的单次实验函数

In [3]:
def run_single_experiment(
    attack_method="textbugger",
    defense_method="none",
    dataset_name="agnews",
    num_examples=1000,

    mode="attack",
    model_path=MODEL_PATH,
    dataset_path=DATASET_PATH,
    results_file="/root/results/joca_04/results.csv",
    attack_log_path="/root/results/joca_04/attack_logs",
    cache_dir=CACHE_DIR,
    model_batch_size=MODEL_BATCH_SIZE,
    max_seq_length=MAX_SEQ_LENGTH,
    mask_token="<unk>",
    mask_rate=MASK_RATE,
    attack_query_budget=100,

    ahp_num_candidates=ENSEMBLE_SIZE,
    ahp_pruning_method="none",
    ahp_pruning_threshold=500,
    ahp_aggregation_strategy="majority_vote",
    ahp_masking_strategy="stochastic",
    ahp_temperature=1.0,

    selfdenoise_ensemble_size=ENSEMBLE_SIZE,
    selfdenoise_denoiser="roberta",

    topk_ensemble_size=ENSEMBLE_SIZE,

    seed=123,
    device=None,
    log_level="INFO",
):
    # 沿用实验 3 的 AHPSettings + ExperimentRunner 调用方式。

    logging.info(
        f"--- 开始实验: 数据集={dataset_name}, 防御={defense_method}, "
        f"攻击={attack_method if mode == 'attack' else 'None'}, seed={seed} ---"
    )

    os.makedirs(os.path.dirname(results_file), exist_ok=True)
    os.makedirs(attack_log_path, exist_ok=True)

    args_list = [
        "--mode", mode,
        "--dataset_name", dataset_name,
        "--num_examples", str(num_examples),
        "--model_path", model_path,
        "--dataset_path", dataset_path,
        "--results_file", results_file,
        "--attack_log_path", attack_log_path,
        "--cache_dir", cache_dir,
        "--model_batch_size", str(model_batch_size),
        "--max_seq_length", str(max_seq_length),
        "--mask_token", mask_token,
        "--mask_rate", str(mask_rate),
        "--defense_method", defense_method,
        "--seed", str(seed),
        "--log_level", log_level,
    ]

    if device:
        args_list.extend(["--device", device])

    if mode == "attack":
        args_list.extend([
            "--attack_method", attack_method,
            "--attack_query_budget", str(attack_query_budget),
        ])

    if defense_method == "ahp":
        args_list.extend([
            "--ahp_num_candidates", str(ahp_num_candidates),
            "--ahp_pruning_method", ahp_pruning_method,
            "--ahp_pruning_threshold", str(ahp_pruning_threshold),
            "--ahp_aggregation_strategy", ahp_aggregation_strategy,
            "--ahp_masking_strategy", ahp_masking_strategy,
            "--ahp_temperature", str(ahp_temperature),
        ])
    elif defense_method == "selfdenoise":
        args_list.extend([
            "--selfdenoise_ensemble_size", str(selfdenoise_ensemble_size),
            "--selfdenoise_denoiser", selfdenoise_denoiser,
        ])
    elif defense_method == "topk":
        args_list.extend([
            "--topk_ensemble_size", str(topk_ensemble_size),
        ])

    print(
        f"dataset={dataset_name}, defense={defense_method}, "
        f"attack={attack_method}, seed={seed}, num={num_examples}, "
        f"max_seq_length={max_seq_length}"
    )

    runner = None
    try:
        args = AHPSettings().parse_args(args_list)
        runner = ExperimentRunner(args)
        runner.run()
        logging.info("--- 实验完成 ---")
        return True
    except Exception as e:
        logging.error("--- 实验失败 ---", exc_info=True)
        print("错误信息:", e)
        return False
    finally:
        if runner is not None:
            del runner
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 3. 攻击统计代码检查与可选补丁

当前攻击结果需要保存：

- `avg_perturbed_words`
- `avg_queries`
- `conditional_robust_accuracy`
- `overall_robust_accuracy`

若此前已经手动修复，可以只运行检查单元，不执行补丁。

In [4]:
EXPERIMENT_RUNNER_FILE = "/root/src/experiment_runner.py"

def attack_metrics_patch_exists():
    with open(EXPERIMENT_RUNNER_FILE, "r", encoding="utf-8") as f:
        source = f.read()

    attack_pos = source.find("    def attack(self):")
    results_pos = source.find("        results_summary = {", attack_pos)
    active_region = source[attack_pos:results_pos]

    return (
        "avg_perturbed_words =" in active_region
        and "avg_queries =" in active_region
        and "conditional_robust_accuracy =" in active_region
        and "overall_robust_accuracy =" in active_region
    )

print("攻击统计代码是否完整:", attack_metrics_patch_exists())

攻击统计代码是否完整: True


In [5]:
# 只有上一格输出 False 时才执行一次。
APPLY_ATTACK_METRICS_PATCH = False

def apply_attack_metrics_patch():
    marker = "# JOCA_ATTACK_METRICS_PATCH_V2"

    with open(EXPERIMENT_RUNNER_FILE, "r", encoding="utf-8") as f:
        source = f.read()

    if marker in source:
        print("补丁已经存在，不重复执行。")
        return

    attack_pos = source.find("    def attack(self):")
    insert_pos = source.find("        results_summary = {", attack_pos)

    if attack_pos < 0 or insert_pos < 0:
        raise RuntimeError("没有找到 attack() 或 results_summary 插入位置。")

    patch_code = '''
        # JOCA_ATTACK_METRICS_PATCH_V2
        perturbed_word_counts = []
        query_counts = []

        for result in results:
            status = result.perturbed_result.goal_status

            if status != GoalFunctionResultStatus.SKIPPED:
                query_value = getattr(result, "num_queries", None)
                if query_value is not None:
                    query_counts.append(float(query_value))

            if status == GoalFunctionResultStatus.SUCCEEDED:
                try:
                    changed_words = (
                        result.original_result.attacked_text.all_words_diff(
                            result.perturbed_result.attacked_text
                        )
                    )
                    perturbed_word_counts.append(len(changed_words))
                except Exception as metric_error:
                    logging.warning(
                        "无法计算扰动词数: %s",
                        metric_error,
                    )

        avg_perturbed_words = (
            float(np.mean(perturbed_word_counts))
            if perturbed_word_counts
            else 0.0
        )
        avg_queries = (
            float(np.mean(query_counts))
            if query_counts
            else 0.0
        )

'''

    backup_file = (
        EXPERIMENT_RUNNER_FILE
        + ".bak_joca_"
        + time.strftime("%Y%m%d_%H%M%S")
    )
    shutil.copy2(EXPERIMENT_RUNNER_FILE, backup_file)

    patched_source = source[:insert_pos] + patch_code + source[insert_pos:]
    compile(patched_source, EXPERIMENT_RUNNER_FILE, "exec")

    with open(EXPERIMENT_RUNNER_FILE, "w", encoding="utf-8") as f:
        f.write(patched_source)

    print("补丁完成。备份文件:", backup_file)

if APPLY_ATTACK_METRICS_PATCH:
    apply_attack_metrics_patch()
else:
    print("未执行补丁。")

未执行补丁。


## 4. 防止重复运行的辅助函数

结果文件按实验类型分开保存。若相同配置已存在，则默认跳过。

In [6]:
def result_already_exists(
    results_file,
    dataset,
    defense,
    attack,
    seed,
    mask_rate=None,
    temperature=None,
    ensemble_size=None,
):
    if not os.path.exists(results_file):
        return False

    try:
        df = pd.read_csv(results_file)
    except Exception:
        return False

    required = ["dataset", "defense", "attack", "seed"]
    if any(column not in df.columns for column in required):
        return False

    mask = (
        (df["dataset"].astype(str) == str(dataset))
        & (df["defense"].astype(str) == str(defense))
        & (df["attack"].astype(str) == str(attack))
        & (df["seed"].astype(int) == int(seed))
    )

    if mask_rate is not None and "mask_rate" in df.columns:
        mask &= np.isclose(
            pd.to_numeric(df["mask_rate"], errors="coerce"),
            float(mask_rate),
            equal_nan=False,
        )

    if temperature is not None and "ahp_temperature" in df.columns:
        mask &= np.isclose(
            pd.to_numeric(df["ahp_temperature"], errors="coerce"),
            float(temperature),
            equal_nan=False,
        )

    if ensemble_size is not None and "ensemble_size" in df.columns:
        mask &= (
            pd.to_numeric(df["ensemble_size"], errors="coerce")
            == int(ensemble_size)
        )

    return bool(mask.any())


def run_or_skip(results_file, config):
    exists = result_already_exists(
        results_file=results_file,
        dataset=config["dataset_name"],
        defense=config["defense_method"],
        attack=config.get("attack_method", "none")
        if config["mode"] == "attack"
        else "none",
        seed=config["seed"],
        mask_rate=config.get("mask_rate"),
        temperature=config.get("ahp_temperature")
        if config["defense_method"] == "ahp"
        else None,
        ensemble_size=(
            config.get("ahp_num_candidates")
            if config["defense_method"] == "ahp"
            else config.get("selfdenoise_ensemble_size")
            if config["defense_method"] == "selfdenoise"
            else config.get("topk_ensemble_size")
            if config["defense_method"] == "topk"
            else None
        ),
    )

    if exists:
        print("跳过已存在配置:", config)
        return True

    return run_single_experiment(
        results_file=results_file,
        **config,
    )

## 5. 已完成的无防御干净准确率

以下结果已经在修正标签 Token 和最大长度 512 后得到，无需重跑：

- SST-2：73.5092%
- AG News：71.0000%

In [7]:
existing_standard_results = pd.DataFrame([
    {
        "dataset": "sst2",
        "defense": "none",
        "attack": "none",
        "num_examples": 872,
        "accuracy": 0.735092,
        "seed": 123,
        "max_seq_length": 512,
    },
    {
        "dataset": "agnews",
        "defense": "none",
        "attack": "none",
        "num_examples": 1000,
        "accuracy": 0.710000,
        "seed": 123,
        "max_seq_length": 512,
    },
])
display(existing_standard_results)

,dataset,defense,attack,num_examples,accuracy,seed,max_seq_length
0,sst2,none,none,872,0.735092,123,512
1,agnews,none,none,1000,0.710000,123,512


## 6. 干净准确率补充实验

只重跑：

- Top-K
- SelfDenoise
- AHP

`smoke` 模式只跑 AG News、20 条、seed=123。  
`full` 模式跑两个数据集和 3 个随机种子。

In [8]:
RUN_CLEAN_EXPERIMENTS = False

CLEAN_RESULTS_FILE = os.path.join(
    RESULT_ROOT,
    f"clean_results_{EXPERIMENT_SCALE}.csv",
)

if RUN_CLEAN_EXPERIMENTS:
    if EXPERIMENT_SCALE == "smoke":
        clean_datasets = ["agnews"]
        clean_seeds = [123]
    else:
        clean_datasets = ["sst2", "agnews"]
        clean_seeds = SEEDS

    clean_failed = []

    for dataset_name in clean_datasets:
        for defense_method in ["topk", "selfdenoise", "ahp"]:
            for seed in clean_seeds:
                config = {
                    "mode": "evaluate",
                    "attack_method": "none",
                    "defense_method": defense_method,
                    "dataset_name": dataset_name,
                    "num_examples": get_num_examples(dataset_name),
                    "dataset_path": DATASET_PATH,
                    "model_path": MODEL_PATH,
                    "cache_dir": CACHE_DIR,
                    "attack_log_path": os.path.join(
                        RESULT_ROOT,
                        "clean_logs",
                        f"{dataset_name}_{defense_method}_seed{seed}",
                    ),
                    "model_batch_size": MODEL_BATCH_SIZE,
                    "max_seq_length": MAX_SEQ_LENGTH,
                    "mask_rate": MASK_RATE,
                    "ahp_num_candidates": ENSEMBLE_SIZE,
                    "ahp_temperature": 1.0,
                    "ahp_pruning_method": "none",
                    "ahp_aggregation_strategy": "majority_vote",
                    "ahp_masking_strategy": "stochastic",
                    "selfdenoise_ensemble_size": ENSEMBLE_SIZE,
                    "selfdenoise_denoiser": "roberta",
                    "topk_ensemble_size": ENSEMBLE_SIZE,
                    "seed": seed,
                    "log_level": "INFO",
                }

                success = run_or_skip(CLEAN_RESULTS_FILE, config)
                if not success:
                    clean_failed.append(config)

    print("失败任务数:", len(clean_failed))
else:
    print("干净准确率实验未启用。")

干净准确率实验未启用。


In [9]:
def summarize_clean_results():
    frames = [existing_standard_results.copy()]

    full_file = os.path.join(RESULT_ROOT, "clean_results_full.csv")
    if os.path.exists(full_file):
        frames.append(pd.read_csv(full_file))

    df = pd.concat(frames, ignore_index=True, sort=False)

    summary = (
        df.groupby(["dataset", "defense"])["accuracy"]
        .agg(["mean", "std", "min", "max", "count"])
        .reset_index()
    )
    display(summary)

    output_file = os.path.join(
        RESULT_ROOT,
        "clean_summary_for_paper.csv",
    )
    summary.to_csv(output_file, index=False)
    print("已保存:", output_file)
    return summary

# 完整实验结束后运行：
# clean_summary = summarize_clean_results()

## 7. 主对抗实验

正式组合：

- 2 个数据集
- 3 种攻击：DeepWordBug、Pruthi、PWWS
- 4 种防御：none、Top-K、SelfDenoise、AHP
- none 只运行 seed=123
- 其余防御运行 42、123、666

`smoke` 模式只测试：

- AG News
- Pruthi
- none 与 AHP
- 10 条样本

In [10]:
RUN_MAIN_ATTACK_EXPERIMENTS = False

MAIN_ATTACK_RESULTS_FILE = os.path.join(
    RESULT_ROOT,
    f"main_attack_results_{EXPERIMENT_SCALE}.csv",
)

if RUN_MAIN_ATTACK_EXPERIMENTS:
    if not attack_metrics_patch_exists():
        raise RuntimeError(
            "攻击统计代码不完整，请先执行第 3 节补丁。"
        )

    if EXPERIMENT_SCALE == "smoke":
        attack_datasets = ["agnews"]
        attack_methods = ["pruthi"]
        defense_seed_map = {
            "none": [123],
            "ahp": [123],
        }
        smoke_num_examples = 10
    else:
        attack_datasets = [ "sst2","agnews"]
        # 
        attack_methods = [ "deepwordbug", "pruthi","pwws"]
        # 
        defense_seed_map = {
            # "none": [123],
            # "topk": SEEDS,
            "selfdenoise": SEEDS,
            # "ahp": SEEDS,
        }
        smoke_num_examples = None

    attack_failed = []

    for dataset_name in attack_datasets:
        query_budget = 100 if dataset_name == "sst2" else 200
        num_examples = (
            smoke_num_examples
            if EXPERIMENT_SCALE == "smoke"
            else FULL_ATTACK_NUM_EXAMPLES[dataset_name]
        )

        for attack_method in attack_methods:
            for defense_method, seeds in defense_seed_map.items():
                for seed in seeds:
                    log_dir = os.path.join(
                        RESULT_ROOT,
                        "main_attack_logs",
                        EXPERIMENT_SCALE,
                        f"{dataset_name}_{attack_method}_"
                        f"{defense_method}_seed{seed}",
                    )

                    config = {
                        "mode": "attack",
                        "attack_method": attack_method,
                        "defense_method": defense_method,
                        "dataset_name": dataset_name,
                        "num_examples": num_examples,
                        "dataset_path": DATASET_PATH,
                        "model_path": MODEL_PATH,
                        "cache_dir": CACHE_DIR,
                        "attack_log_path": log_dir,
                        "model_batch_size": MODEL_BATCH_SIZE,
                        "max_seq_length": MAX_SEQ_LENGTH,
                        "mask_rate": MASK_RATE,
                        "attack_query_budget": query_budget,
                        "ahp_num_candidates": ENSEMBLE_SIZE,
                        "ahp_temperature": 1.0,
                        "ahp_pruning_method": "none",
                        "ahp_aggregation_strategy": "majority_vote",
                        "ahp_masking_strategy": "stochastic",
                        "selfdenoise_ensemble_size": ENSEMBLE_SIZE,
                        "selfdenoise_denoiser": "roberta",
                        "topk_ensemble_size": ENSEMBLE_SIZE,
                        "seed": seed,
                        "log_level": "INFO",
                    }

                    success = run_or_skip(
                        MAIN_ATTACK_RESULTS_FILE,
                        config,
                    )
                    if not success:
                        attack_failed.append(config)

    print("失败任务数:", len(attack_failed))
else:
    print("主对抗实验未启用。")

主对抗实验未启用。


In [11]:
def summarize_main_attack_results():
    results_file = os.path.join(
        RESULT_ROOT,
        "main_attack_results_full.csv",
    )
    if not os.path.exists(results_file):
        raise FileNotFoundError(results_file)

    df = pd.read_csv(results_file)

    metrics = [
        column
        for column in [
            "clean_accuracy_on_attack_set",
            "conditional_robust_accuracy",
            "overall_robust_accuracy",
            "attack_success_rate",
            "avg_queries",
            "avg_perturbed_words",
        ]
        if column in df.columns
    ]

    summary = (
        df.groupby(
            ["dataset", "attack", "defense"]
        )[metrics]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    summary.columns = [
        "_".join(
            [str(item) for item in column if str(item)]
        )
        if isinstance(column, tuple)
        else str(column)
        for column in summary.columns
    ]

    display(summary)

    output_file = os.path.join(
        RESULT_ROOT,
        "main_attack_summary_for_paper.csv",
    )
    summary.to_csv(output_file, index=False)
    print("已保存:", output_file)
    return summary

# 完整实验结束后运行：
# main_attack_summary = summarize_main_attack_results()

## 8. AHP 核心消融实验

固定：

- AG News
- Pruthi
- AHP
- 3 个随机种子
- 查询预算 200

分别改变：

- 温度：0.1、0.5、1.0、2.0、5.0
- 候选数：5、10、20、50
- 掩码率：0.05、0.10、0.15、0.20、0.25

`smoke` 模式仅运行一个基准配置。

In [12]:
RUN_ABLATION_EXPERIMENTS = True

ABLATION_RESULTS_FILE = os.path.join(
    RESULT_ROOT,
    f"ablation_results_{EXPERIMENT_SCALE}.csv",
)
ABLATION_INDEX_FILE = os.path.join(
    RESULT_ROOT,
    f"ablation_index_{EXPERIMENT_SCALE}.csv",
)

def add_unique_ablation_config(
    configs,
    memberships,
    sweep_name,
    sweep_value,
    temperature,
    candidates,
    mask_rate,
    seed,
):
    key = (
        float(temperature),
        int(candidates),
        float(mask_rate),
        int(seed),
    )

    if key not in configs:
        configs[key] = {
            "mode": "attack",
            "attack_method": "pruthi",
            "defense_method": "ahp",
            "dataset_name": "agnews",
            "num_examples": (
                10
                if EXPERIMENT_SCALE == "smoke"
                else FULL_ABLATION_NUM_EXAMPLES
            ),
            "dataset_path": DATASET_PATH,
            "model_path": MODEL_PATH,
            "cache_dir": CACHE_DIR,
            "attack_log_path": os.path.join(
                RESULT_ROOT,
                "ablation_logs",
                EXPERIMENT_SCALE,
                f"t{temperature}_m{candidates}_"
                f"r{mask_rate}_seed{seed}",
            ),
            "model_batch_size": MODEL_BATCH_SIZE,
            "max_seq_length": MAX_SEQ_LENGTH,
            "mask_rate": mask_rate,
            "attack_query_budget": 200,
            "ahp_num_candidates": candidates,
            "ahp_temperature": temperature,
            "ahp_pruning_method": "none",
            "ahp_aggregation_strategy": "majority_vote",
            "ahp_masking_strategy": "stochastic",
            "seed": seed,
            "log_level": "INFO",
        }

    memberships.append({
        "temperature": temperature,
        "ensemble_size": candidates,
        "mask_rate": mask_rate,
        "seed": seed,
        "sweep_name": sweep_name,
        "sweep_value": sweep_value,
    })


def build_ablation_configs():
    configs = {}
    memberships = []

    seeds = [123] if EXPERIMENT_SCALE == "smoke" else SEEDS

    if EXPERIMENT_SCALE == "smoke":
        add_unique_ablation_config(
            configs,
            memberships,
            "smoke",
            "baseline",
            1.0,
            5,
            0.15,
            123,
        )
        return list(configs.values()), memberships

    for seed in seeds:
        # for value in [0.1, 0.5, 1.0, 2.0, 5.0]:
        #     add_unique_ablation_config(
        #         configs, memberships,
        #         "temperature", value,
        #         value, 50, 0.15, seed,
        #     )

        # for value in [5, 10, 20, 50]:
        #     add_unique_ablation_config(
        #         configs, memberships,
        #         "ensemble_size", value,
        #         1.0, value, 0.15, seed,
        #     )
# 0.05, 0.10,
        for value in [ 0.15, 0.20, 0.25]:
            add_unique_ablation_config(
                configs, memberships,
                "mask_rate", value,
                1.0, 50, value, seed,
            )

    return list(configs.values()), memberships


if RUN_ABLATION_EXPERIMENTS:
    if not attack_metrics_patch_exists():
        raise RuntimeError(
            "攻击统计代码不完整，请先执行第 3 节补丁。"
        )

    ablation_configs, ablation_memberships = (
        build_ablation_configs()
    )

    pd.DataFrame(ablation_memberships).to_csv(
        ABLATION_INDEX_FILE,
        index=False,
    )

    ablation_failed = []
    for config in ablation_configs:
        success = run_or_skip(ABLATION_RESULTS_FILE, config)
        if not success:
            ablation_failed.append(config)

    print("实际去重后运行任务数:", len(ablation_configs))
    print("失败任务数:", len(ablation_failed))
else:
    print("消融实验未启用。")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

跳过已存在配置: {'mode': 'attack', 'attack_method': 'pruthi', 'defense_method': 'ahp', 'dataset_name': 'agnews', 'num_examples': 50, 'dataset_path': '/root/dataset', 'model_path': '/root/autodl-tmp/alpaca-native', 'cache_dir': '/root/autodl-tmp/cache', 'attack_log_path': '/root/results/joca_04/ablation_logs/full/t1.0_m50_r0.15_seed123', 'model_batch_size': 4, 'max_seq_length': 512, 'mask_rate': 0.15, 'attack_query_budget': 200, 'ahp_num_candidates': 50, 'ahp_temperature': 1.0, 'ahp_pruning_method': 'none', 'ahp_aggregation_strategy': 'majority_vote', 'ahp_masking_strategy': 'stochastic', 'seed': 123, 'log_level': 'INFO'}
dataset=agnews, defense=ahp, attack=pruthi, seed=123, num=50, max_seq_length=512


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

textattack: Unknown if model of class <class 'src.models.model_loader.AlpacaModel'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
[Succeeded / Failed / Skipped / Total] 8 / 29 / 13 / 50: 100%|██████████| 50/50 [1:27:15<00:00, 104.71s/it]

...

dataset=agnews, defense=ahp, attack=pruthi, seed=123, num=50, max_seq_length=512


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

textattack: Unknown if model of class <class 'src.models.model_loader.AlpacaModel'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
[Succeeded / Failed / Skipped / Total] 10 / 28 / 12 / 50: 100%|██████████| 50/50 [1:30:28<00:00, 108.57s/it]

正在攻击:   0%|                                                              | 0/50 [00:00<?, ?it/s]

实际去重后运行任务数: 3
失败任务数: 0


In [13]:
def summarize_ablation_results():
    results_file = os.path.join(
        RESULT_ROOT,
        "ablation_results_full.csv",
    )
    index_file = os.path.join(
        RESULT_ROOT,
        "ablation_index_full.csv",
    )

    if not os.path.exists(results_file):
        raise FileNotFoundError(results_file)
    if not os.path.exists(index_file):
        raise FileNotFoundError(index_file)

    results = pd.read_csv(results_file)
    index_df = pd.read_csv(index_file)

    merge_columns = [
        "seed",
        "mask_rate",
        "ahp_temperature",
        "ensemble_size",
    ]

    index_df = index_df.rename(
        columns={"temperature": "ahp_temperature"}
    )

    merged = index_df.merge(
        results,
        on=merge_columns,
        how="inner",
    )

    metrics = [
        column
        for column in [
            "conditional_robust_accuracy",
            "overall_robust_accuracy",
            "attack_success_rate",
            "avg_queries",
        ]
        if column in merged.columns
    ]

    summary = (
        merged.groupby(
            ["sweep_name", "sweep_value"]
        )[metrics]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    summary.columns = [
        "_".join(
            [str(item) for item in column if str(item)]
        )
        if isinstance(column, tuple)
        else str(column)
        for column in summary.columns
    ]

    display(summary)

    output_file = os.path.join(
        RESULT_ROOT,
        "ablation_summary_for_paper.csv",
    )
    summary.to_csv(output_file, index=False)
    print("已保存:", output_file)
    return summary

# 完整实验结束后运行：
ablation_summary = summarize_ablation_results()

,sweep_name,sweep_value,conditional_robust_accuracy_mean,conditional_robust_accuracy_std,conditional_robust_accuracy_count,overall_robust_accuracy_mean,overall_robust_accuracy_std,overall_robust_accuracy_count,attack_success_rate_mean,attack_success_rate_std,attack_success_rate_count,avg_queries_mean,avg_queries_std,avg_queries_count
0,mask_rate,0.15,0.757576,NaN,1,0.50,NaN,1,0.242424,NaN,1,200.0,NaN,1
1,mask_rate,0.20,0.783784,NaN,1,0.58,NaN,1,0.216216,NaN,1,200.0,NaN,1
2,mask_rate,0.25,0.736842,NaN,1,0.56,NaN,1,0.263158,NaN,1,200.0,NaN,1


已保存: /root/results/joca_04/ablation_summary_for_paper.csv


## 9. 扰动位置覆盖率实验

该实验复用主实验中无防御方法生成的攻击日志，不重新运行 TextAttack。

比较：

- Random
- Top-K
- AHP

统计：

- 单候选命中率
- 候选集至少命中一次的比例
- 平均扰动位置覆盖率
- 最优候选覆盖率

优先分析 AG News 上的 Pruthi 和 DeepWordBug。

In [14]:
ANSI_PATTERN = re.compile(r"\x1b\[[0-9;]*m")
MARK_PATTERN = re.compile(r"\[\[(.*?)\]\]")

def clean_attack_text(value):
    text = "" if pd.isna(value) else str(value)
    text = ANSI_PATTERN.sub("", text)
    text = MARK_PATTERN.sub(r"\1", text)
    return text.strip()


def find_csv_column(df, possible_names):
    lower_map = {
        str(column).strip().lower(): column
        for column in df.columns
    }
    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    raise KeyError(
        f"没有找到列 {possible_names}，当前列为 {list(df.columns)}"
    )


def changed_word_indices(original_text, perturbed_text):
    original_words = original_text.split()
    perturbed_words = perturbed_text.split()

    matcher = difflib.SequenceMatcher(
        a=original_words,
        b=perturbed_words,
        autojunk=False,
    )

    changed = set()
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            changed.update(range(j1, j2))
            if j1 == j2 and perturbed_words:
                changed.add(
                    min(j1, len(perturbed_words) - 1)
                )
    return changed


def load_successful_attack_pairs(csv_file):
    df = pd.read_csv(csv_file)

    original_col = find_csv_column(
        df,
        ["original_text", "original text", "original"],
    )
    perturbed_col = find_csv_column(
        df,
        ["perturbed_text", "perturbed text", "perturbed"],
    )

    result_col = None
    for names in [
        ["result_type", "result type"],
        ["result", "status"],
    ]:
        try:
            result_col = find_csv_column(df, names)
            break
        except KeyError:
            pass

    pairs = pd.DataFrame({
        "original_text": df[original_col].map(
            clean_attack_text
        ),
        "perturbed_text": df[perturbed_col].map(
            clean_attack_text
        ),
    })

    if result_col is not None:
        success_mask = (
            df[result_col]
            .astype(str)
            .str.lower()
            .str.contains("success", na=False)
        )
        pairs = pairs[success_mask].copy()

    return pairs.reset_index(drop=True)


def locate_none_attack_log(attack_name):
    log_root = os.path.join(
        RESULT_ROOT,
        "main_attack_logs",
        "full",
        f"agnews_{attack_name}_none_seed123",
    )

    if not os.path.exists(log_root):
        raise FileNotFoundError(log_root)

    csv_files = [
        os.path.join(log_root, name)
        for name in os.listdir(log_root)
        if name.endswith(".csv")
    ]

    if not csv_files:
        raise FileNotFoundError(
            f"{log_root} 中没有 CSV 日志。"
        )

    return csv_files[0]


def calculate_coverage_metrics(
    perturb_indices,
    mask_sets,
):
    if not perturb_indices:
        return None

    hits = [
        len(mask_indices & perturb_indices) > 0
        for mask_indices in mask_sets
    ]
    recalls = [
        len(mask_indices & perturb_indices)
        / len(perturb_indices)
        for mask_indices in mask_sets
    ]

    return {
        "candidate_hit_rate": float(np.mean(hits)),
        "ensemble_hit_rate": float(any(hits)),
        "mean_perturbation_recall": float(np.mean(recalls)),
        "best_candidate_recall": float(np.max(recalls)),
    }

In [15]:
RUN_COVERAGE_EXPERIMENT = False

def run_coverage_experiment(
    max_samples_per_attack=100,
    num_candidates=50,
    seed=123,
):
    args_list = [
        "--mode", "evaluate",
        "--dataset_name", "agnews",
        "--dataset_path", DATASET_PATH,
        "--num_examples", "1",
        "--model_path", MODEL_PATH,
        "--cache_dir", CACHE_DIR,
        "--defense_method", "ahp",
        "--model_batch_size", str(MODEL_BATCH_SIZE),
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--mask_rate", str(MASK_RATE),
        "--ahp_num_candidates", str(num_candidates),
        "--ahp_temperature", "1.0",
        "--ahp_masking_strategy", "stochastic",
        "--ahp_pruning_method", "none",
        "--ahp_aggregation_strategy", "majority_vote",
        "--seed", str(seed),
        "--results_file",
        os.path.join(RESULT_ROOT, "coverage_unused.csv"),
        "--attack_log_path",
        os.path.join(RESULT_ROOT, "coverage_unused_logs"),
    ]

    args = AHPSettings().parse_args(args_list)
    runner = ExperimentRunner(args)
    masker = runner.alpaca_model.adversarial_masker
    rng = np.random.default_rng(seed)

    detail_rows = []

    try:
        for attack_name in ["pruthi", "deepwordbug"]:
            attack_log = locate_none_attack_log(attack_name)
            pairs = load_successful_attack_pairs(
                attack_log
            ).head(max_samples_per_attack)

            print(
                attack_name,
                "成功攻击样本数:",
                len(pairs),
            )

            for row_index, row in pairs.iterrows():
                adversarial_text = row["perturbed_text"]
                words = adversarial_text.split()
                word_count = len(words)

                if word_count == 0:
                    continue

                perturb_indices = changed_word_indices(
                    row["original_text"],
                    adversarial_text,
                )
                if not perturb_indices:
                    continue

                n_mask = max(
                    1,
                    int(round(word_count * MASK_RATE)),
                )
                n_mask = min(n_mask, word_count)

                importance = (
                    masker._calculate_word_importance(
                        adversarial_text
                    )
                )
                if len(importance) != word_count:
                    continue

                topk_set = set(
                    np.argsort(importance)[::-1][
                        :n_mask
                    ].tolist()
                )

                scores = np.asarray(
                    importance,
                    dtype=float,
                )
                if scores.max() > 0:
                    scores = scores / scores.max()

                exp_scores = np.exp(scores / 1.0)
                probabilities = (
                    exp_scores / exp_scores.sum()
                )

                ahp_sets = [
                    set(
                        rng.choice(
                            word_count,
                            size=n_mask,
                            replace=False,
                            p=probabilities,
                        ).tolist()
                    )
                    for _ in range(num_candidates)
                ]

                random_sets = [
                    set(
                        rng.choice(
                            word_count,
                            size=n_mask,
                            replace=False,
                        ).tolist()
                    )
                    for _ in range(num_candidates)
                ]

                methods = {
                    "Random": random_sets,
                    "Top-K": [topk_set],
                    "AHP": ahp_sets,
                }

                if word_count <= 20:
                    length_group = "1-20"
                elif word_count <= 40:
                    length_group = "21-40"
                elif word_count <= 80:
                    length_group = "41-80"
                else:
                    length_group = "81+"

                for method, mask_sets in methods.items():
                    metrics = calculate_coverage_metrics(
                        perturb_indices,
                        mask_sets,
                    )
                    if metrics is None:
                        continue

                    detail_rows.append({
                        "attack": attack_name,
                        "sample_index": row_index,
                        "method": method,
                        "word_count": word_count,
                        "length_group": length_group,
                        "num_perturbed_positions": len(
                            perturb_indices
                        ),
                        "num_masked_positions": n_mask,
                        **metrics,
                    })

                if (row_index + 1) % 10 == 0:
                    print(
                        attack_name,
                        "已处理:",
                        row_index + 1,
                    )
    finally:
        del runner
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    details = pd.DataFrame(detail_rows)

    if details.empty:
        raise RuntimeError("没有生成覆盖率结果。")

    summary = (
        details.groupby(
            ["attack", "method", "length_group"]
        )[
            [
                "candidate_hit_rate",
                "ensemble_hit_rate",
                "mean_perturbation_recall",
                "best_candidate_recall",
            ]
        ]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    summary.columns = [
        "_".join(
            [str(item) for item in column if str(item)]
        )
        if isinstance(column, tuple)
        else str(column)
        for column in summary.columns
    ]

    details_file = os.path.join(
        RESULT_ROOT,
        "coverage_details.csv",
    )
    summary_file = os.path.join(
        RESULT_ROOT,
        "coverage_summary_for_paper.csv",
    )

    details.to_csv(details_file, index=False)
    summary.to_csv(summary_file, index=False)

    display(summary)
    print("已保存:", details_file)
    print("已保存:", summary_file)

    return details, summary


if RUN_COVERAGE_EXPERIMENT:
    coverage_limit = (
        10
        if EXPERIMENT_SCALE == "smoke"
        else 100
    )
    coverage_details, coverage_summary = (
        run_coverage_experiment(
            max_samples_per_attack=coverage_limit,
            num_candidates=50,
            seed=123,
        )
    )
else:
    print("扰动位置覆盖率实验未启用。")

扰动位置覆盖率实验未启用。


## 10. 计算开销实验

从每个数据集固定抽取样本，比较：

- none
- Top-K
- SelfDenoise
- AHP

统计：

- 平均单样本延迟
- 吞吐率
- 峰值显存
- 目标模型前向调用次数
- RoBERTa 前向调用次数
- 梯度显著性计算次数

`smoke`：10 条、重复 1 次。  
`full`：100 条、重复 5 次。

In [16]:
RUN_EFFICIENCY_EXPERIMENT = False

def build_runner_for_efficiency(
    dataset_name,
    defense_method,
    num_examples,
    seed,
):
    args_list = [
        "--mode", "evaluate",
        "--dataset_name", dataset_name,
        "--dataset_path", DATASET_PATH,
        "--num_examples", str(num_examples),
        "--model_path", MODEL_PATH,
        "--cache_dir", CACHE_DIR,
        "--defense_method", defense_method,
        "--model_batch_size", str(MODEL_BATCH_SIZE),
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--mask_rate", str(MASK_RATE),
        "--ahp_num_candidates", str(ENSEMBLE_SIZE),
        "--ahp_temperature", "1.0",
        "--ahp_masking_strategy", "stochastic",
        "--ahp_pruning_method", "none",
        "--ahp_aggregation_strategy", "majority_vote",
        "--selfdenoise_ensemble_size",
        str(ENSEMBLE_SIZE),
        "--selfdenoise_denoiser", "roberta",
        "--topk_ensemble_size", str(ENSEMBLE_SIZE),
        "--seed", str(seed),
        "--results_file",
        os.path.join(
            RESULT_ROOT,
            "efficiency_unused.csv",
        ),
        "--attack_log_path",
        os.path.join(
            RESULT_ROOT,
            "efficiency_unused_logs",
        ),
    ]

    args = AHPSettings().parse_args(args_list)
    return ExperimentRunner(args)


def run_efficiency_experiment(
    sample_count=100,
    repeats=5,
    seed=123,
):
    detail_rows = []

    for dataset_name in ["sst2", "agnews"]:
        split = (
            "validation"
            if dataset_name == "sst2"
            else "test"
        )

        raw_data = load_dataset(
            os.path.join(DATASET_PATH, dataset_name),
            dataset_name,
            split=split,
            num_examples=sample_count,
        )
        texts = [item[0] for item in raw_data]

        for defense_method in [
            "none",
            "topk",
            "selfdenoise",
            "ahp",
        ]:
            print(
                "\n计算开销:",
                dataset_name,
                defense_method,
            )

            runner = build_runner_for_efficiency(
                dataset_name,
                defense_method,
                len(texts),
                seed,
            )
            model = runner.alpaca_model

            # 预热并触发 RoBERTa 延迟加载
            model.predict_batch(
                texts[: min(2, len(texts))]
            )

            counters = {
                "target_forward_calls": 0,
                "roberta_forward_calls": 0,
                "gradient_importance_calls": 0,
            }

            def target_forward_hook(
                module,
                module_input,
                module_output,
            ):
                counters["target_forward_calls"] += 1

            target_hook_handle = (
                model.model.register_forward_hook(
                    target_forward_hook
                )
            )

            roberta_hook_handle = None
            if model.roberta_model is not None:
                def roberta_forward_hook(
                    module,
                    module_input,
                    module_output,
                ):
                    counters["roberta_forward_calls"] += 1

                roberta_hook_handle = (
                    model.roberta_model.register_forward_hook(
                        roberta_forward_hook
                    )
                )

            original_importance_method = None
            if model.adversarial_masker is not None:
                original_importance_method = (
                    model.adversarial_masker
                    ._calculate_word_importance
                )

                def counted_importance(text):
                    counters[
                        "gradient_importance_calls"
                    ] += 1
                    return original_importance_method(text)

                model.adversarial_masker._calculate_word_importance = (
                    counted_importance
                )

            try:
                for repeat_index in range(repeats):
                    for counter_name in counters:
                        counters[counter_name] = 0

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        torch.cuda.reset_peak_memory_stats()
                        torch.cuda.synchronize()
                        memory_before = (
                            torch.cuda.memory_allocated()
                        )
                    else:
                        memory_before = 0

                    start_time = time.perf_counter()

                    predictions = model.predict_batch(texts)

                    if torch.cuda.is_available():
                        torch.cuda.synchronize()

                    elapsed = (
                        time.perf_counter() - start_time
                    )

                    if torch.cuda.is_available():
                        peak_memory = (
                            torch.cuda.max_memory_allocated()
                        )
                    else:
                        peak_memory = 0

                    detail_rows.append({
                        "dataset": dataset_name,
                        "defense": defense_method,
                        "repeat": repeat_index + 1,
                        "num_examples": len(texts),
                        "elapsed_seconds": elapsed,
                        "latency_seconds_per_example":
                            elapsed / len(texts),
                        "throughput_examples_per_second":
                            len(texts) / elapsed,
                        "peak_memory_gb":
                            peak_memory / 1024**3,
                        "extra_peak_memory_gb":
                            max(
                                0,
                                peak_memory - memory_before,
                            ) / 1024**3,
                        **counters,
                    })

                    print(
                        f"repeat={repeat_index + 1}, "
                        f"latency={elapsed / len(texts):.4f}s, "
                        f"throughput={len(texts) / elapsed:.4f}/s"
                    )
            finally:
                target_hook_handle.remove()

                if roberta_hook_handle is not None:
                    roberta_hook_handle.remove()

                if (
                    model.adversarial_masker is not None
                    and original_importance_method is not None
                ):
                    model.adversarial_masker._calculate_word_importance = (
                        original_importance_method
                    )

                del runner
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    details = pd.DataFrame(detail_rows)

    summary = (
        details.groupby(
            ["dataset", "defense"]
        )[
            [
                "latency_seconds_per_example",
                "throughput_examples_per_second",
                "peak_memory_gb",
                "extra_peak_memory_gb",
                "target_forward_calls",
                "roberta_forward_calls",
                "gradient_importance_calls",
            ]
        ]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    summary.columns = [
        "_".join(
            [str(item) for item in column if str(item)]
        )
        if isinstance(column, tuple)
        else str(column)
        for column in summary.columns
    ]

    details_file = os.path.join(
        RESULT_ROOT,
        "efficiency_details.csv",
    )
    summary_file = os.path.join(
        RESULT_ROOT,
        "efficiency_summary_for_paper.csv",
    )

    details.to_csv(details_file, index=False)
    summary.to_csv(summary_file, index=False)

    display(summary)
    print("已保存:", details_file)
    print("已保存:", summary_file)

    return details, summary


if RUN_EFFICIENCY_EXPERIMENT:
    efficiency_sample_count = (
        10
        if EXPERIMENT_SCALE == "smoke"
        else 100
    )
    efficiency_repeats = (
        1
        if EXPERIMENT_SCALE == "smoke"
        else 5
    )

    efficiency_details, efficiency_summary = (
        run_efficiency_experiment(
            sample_count=efficiency_sample_count,
            repeats=efficiency_repeats,
            seed=123,
        )
    )
else:
    print("计算开销实验未启用。")

计算开销实验未启用。


## 11. 查看所有生成的结果文件

In [17]:
result_files = []

for root_dir, dir_names, file_names in os.walk(
    RESULT_ROOT
):
    for file_name in file_names:
        if file_name.endswith(
            (".csv", ".log", ".json")
        ):
            result_files.append(
                os.path.join(root_dir, file_name)
            )

result_files = sorted(result_files)

print("结果文件数量:", len(result_files))
for file_name in result_files:
    print(file_name)

结果文件数量: 54
/root/results/joca_04/.ipynb_checkpoints/ablation_index_full-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/ablation_results_full-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/clean_results_full-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/clean_results_smoke-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/main_attack_results_full-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/main_attack_results_smoke-checkpoint.csv
/root/results/joca_04/.ipynb_checkpoints/main_attack_summary_for_paper-checkpoint.csv
/root/results/joca_04/ablation_index_full.csv
/root/results/joca_04/ablation_logs/full/t0.1_m50_r0.15_seed123/agnews_pruthi_ahp_log.csv
/root/results/joca_04/ablation_logs/full/t0.5_m50_r0.15_seed123/agnews_pruthi_ahp_log.csv
/root/results/joca_04/ablation_logs/full/t1.0_m10_r0.15_seed123/agnews_pruthi_ahp_log.csv
/root/results/joca_04/ablation_logs/full/t1.0_m20_r0.15_seed123/agnews_pruthi_ahp_log.csv
/root/results/joca_04/ablatio